In [1]:
import json
import numpy as np
from pathlib import Path
CORPUS_PATH = Path("fallback_corpus.json")
import arxiv
from sentence_transformers import SentenceTransformer, util
import faiss
import os
import anthropic
import re
import load_dotenv

Disclaimer : Several responses were written using Claude Haiku 5.5 Medium based on inputs for better phrasing and consise wording.

In [12]:
def fetch_abstracts(query: str, max_results: int = 48):
    """Fetch paper abstracts from arXiv API based on search query."""
    try:
        client = arxiv.Client()
        search = arxiv.Search(
            query=query,
            max_results=max_results,
            sort_by=arxiv.SortCriterion.Relevance
        )
        results = []
        for paper in client.results(search):
            results.append({
                'id': paper.entry_id.split('/abs/')[-1],
                'title': paper.title,
                'authors': [str(author) for author in paper.authors],
                'abstract': paper.summary,
                'published': str(paper.published),
                'pdf_url': paper.pdf_url
            })
        return results
      # return abstracts
    except Exception as e:
        print(f"Error fetching from arXiv: {e}")
        return []
        
def save_corpus(corpus, filename="corpus.json"):
    """Save the corpus to a JSON file."""
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(corpus, f, indent=2, ensure_ascii=False)

def load_corpus(filename="corpus.json"):
    """Reload the corpus without re-fetching."""
    with open(filename, "r", encoding="utf-8") as f:
        return json.load(f)

def embed_texts(texts):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    # Generate embeddings
    embeddings = model.encode(texts, convert_to_tensor=False)
    embeddings_array = np.array(embeddings)
    return embeddings_array
    
def cosine_similarity(emb1, emb2):
    """Calculate cosine similarity manually."""
    dot_product = np.dot(emb1, emb2)
    magnitude1 = np.linalg.norm(emb1)
    magnitude2 = np.linalg.norm(emb2)
    similarity = dot_product / (magnitude1 * magnitude2)
    return similarity
 
def build_index(embeddings, path="corpus.faiss"):
    """L2-normalize, build a FAISS IndexFlatIP, add all vectors, and save it to disk."""
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-12
    normalized_embeddings = np.array(embeddings / norms, dtype=np.float32)
    index = faiss.IndexFlatIP(normalized_embeddings.shape[1])
    index.add(normalized_embeddings)
    faiss.write_index(index, path)   # <-- new
    return index
 
def retrieve(query: str, corpus, index, k: int = 3):
    """Embed query and retrieve top-k by cosine similarity."""
    q_vec = embed_texts([query])[0]
    q_vec = q_vec / (np.linalg.norm(q_vec) + 1e-12)
    q_vec = np.array(q_vec, dtype=np.float32).reshape(1, -1)  # FAISS needs shape (1, d), float32

    scores, indices = index.search(q_vec, k)

    return [
        (corpus[i], float(score))
        for score, i in zip(scores[0], indices[0])
        if i != -1  # FAISS returns -1 if fewer than k results exist
    ]
def query_llm(prompt: str, model: str = "claude-haiku-4-5-20251001",
              max_tokens: int = 512, system: str | None = None) -> str:
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        raise RuntimeError("Set ANTHROPIC_API_KEY first.")
    client = anthropic.Anthropic(api_key=api_key)
    kwargs = dict(model=model, max_tokens=max_tokens,
                  messages=[{"role": "user", "content": prompt}])
    if system:
        kwargs["system"] = system
    msg = client.messages.create(**kwargs)
    return "".join(b.text for b in msg.content if b.type == "text")
    

def answer_with_rag(question: str, retrieved):
    context = "\n\n".join(f"[{p['id']}] {p['title']}\n{p['abstract']}" for p, _ in retrieved)
    prompt = f"""Answer the physics question using ONLY the context below.
- Cite the arXiv ID in square brackets after each claim.
- Support each claim with a short exact quote from the context in double quotes.
- If the context does not contain the answer, reply exactly: "The context does not contain the answer."
- Do not use outside knowledge.

Context:
{context}

Question: {question}

Answer:"""
    answer = query_llm(prompt)
    return {"answer": answer,
            "sources": [p["id"] for p, _ in retrieved],
            "verification": verify_answer(answer, retrieved)}

def verify_answer(answer: str, retrieved):
    retrieved_ids = {p["id"] for p, _ in retrieved}
    cited = {c.strip() for c in re.findall(r"\[([^\[\]]+?)\]", answer)}
    texts = [" ".join(p["abstract"].split()).lower() for p, _ in retrieved]
    quotes = re.findall(r'"([^"]{15,})"', answer)
    found = [any(" ".join(q.split()).lower() in t for t in texts) for q in quotes]
    return {"cited_ids": sorted(cited),
            "hallucinated_citations": sorted(cited - retrieved_ids),
            "citations_ok": bool(cited) and cited <= retrieved_ids,
            "quotes_verbatim": all(found) if quotes else False,
            "refused": "does not contain the answer" in answer.lower()}
def canon(x):
    """Canonical arXiv ID: strip URL prefix, strip version suffix, trim whitespace."""
    x = x.strip().split("/abs/")[-1]
    return re.sub(r"v\d+$", "", x)

1. Fetch abstracts:

In [13]:
if __name__ == "__main__":
    filename = "corpus.json"
     # 1-2. Get the data (only cache it if the fetch actually returned papers)
    if os.path.exists(filename):
        corpus = load_corpus(filename)
    else:
        corpus = fetch_abstracts("Isothermal Prolate Core Models in Filamentary Molecular Clouds", max_results=35)
        save_corpus(corpus, filename)
    print(f"Number of results retrieved: {len(corpus)}")
    if corpus:
        print(f"Title of first abstract: {corpus[0]['title']}")

Number of results retrieved: 35
Title of first abstract: A U-band survey of brown dwarfs in the Taurus Molecular Cloud with the XMM-Newton Optical/UV Monitor


2. Embed the abstracts, indexing and similarity 

In [14]:
abstracts = [p["abstract"] for p in corpus]
embeddings = embed_texts(abstracts)
print("Embedding shape:", embeddings.shape)   # e.g. (35, 384)
index = build_index(embeddings)
print("Vectors in index:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (35, 384)
Vectors in index: 35


In [15]:
# Check similarity of two related papers
for i, p in enumerate(corpus):
    print(i, p["title"])          # read these and pick two related ones
a, b = 1, 2                     # change to the indices you pick
sim = cosine_similarity(embeddings[a], embeddings[b])
print(f"Cosine similarity between [{a}] and [{b}]: {sim:.4f}", "PASS" if sim > 0.6 else "below 0.6, pick a closer pair")

0 A U-band survey of brown dwarfs in the Taurus Molecular Cloud with the XMM-Newton Optical/UV Monitor
1 Prolate Cores in Filamentary Molecular Clouds
2 Gravitational instability of filamentary molecular clouds, including ambipolar diffusion; Non-isothermal filament
3 Shapes of Molecular Cloud Cores and the Filamentary Mode of Star Formation
4 Evolution of Prolate Molecular Clouds at HII Boundaries: I. Formation of fragment-core structures
5 A New Model for Filamentary Molecular Clouds
6 Collapse and Fragmentation of Magnetic Molecular Cloud Cores with the Enzo AMR MHD Code. II. Prolate and Oblate Cores
7 Helical Fields and Filamentary Molecular Clouds
8 Gravitational instability of non-isothermal filamentary molecular clouds, in presence of external pressure
9 Collapse and Fragmentation of Molecular Cloud Cores. X. Magnetic Braking of Prolate and Oblate Cores
10 Effect of magnetic field on the rotating filamentary molecular clouds
11 Properties of the dense cores and filamentary struc

3. Design your evaluation set 

In [22]:
question1 = " How much is the Bonnor-Ebert critical mass is reduced by when the helical fields are threaded throughout the core models?"
answer1_manual_search = "20%" #1 json

question2 = "How does the density distributions fall off as a function of radius for isothermal filament models with helical magnetic feilds?"
answer2_manual_search = "-1.8 to -2" #5  json

question3 = "When the magnetic field is initially uniform and aligned with the cloud rotation axes, what are the initial ratios of rotational energy to gravitational energy? "
answer3_manual_search = "from $10^{-4}$ to 0.1" #6 json

question4 = "How do Prolate cores with central densities 20 times higher than their boundary densities compare to prolate cores with central densities 100 times higher? "
# 9

question5 = "What is the temperature range for the N49 bubble site?"
#23

In [24]:
EVAL_SET = [
    {
        "question": question1,       # "20%"
        "gold_ids": ["astro-ph/9909356v1"],           # Prolate Cores in Filamentary Molecular Clouds
        "use_for_precision": True,
    },
    {
        "question": question2,     # "-1.8 to -2"
        "gold_ids": ["astro-ph/9905148v2"],            # A New Model for Filamentary Molecular Clouds 
        "use_for_precision": True,
    },
    {
        "question": question3,      # "from $10^{-4}$ to 0.1"
        "gold_ids": ["1408.2479v1"],                   # Collapse and Fragmentation ... II. Prolate and Oblate Cores
        "use_for_precision": True,
    },
    {
        "question": question4,
        "gold_ids": ["0903.1810v1"],                   # Collapse and Fragmentation ... X. Magnetic Braking
        "use_for_precision": False,                   
    },
    {
        "question": question5,
        "gold_ids": ["1711.06412v1"],                  # New insights in the mid-infrared bubble N49 site
        "use_for_precision": False,
    },
]


corpus_canon = {canon(p["id"]) for p in corpus}
for i, e in enumerate(EVAL_SET):
    e["gold_canon"] = {canon(g) for g in e["gold_ids"]}
    missing = e["gold_canon"] - corpus_canon
    assert not missing, f"Q{i+1} gold not in corpus: {missing}"
print("All gold IDs found in corpus.")

GOLD = {i: e["gold_canon"] for i, e in enumerate(EVAL_SET) if e["use_for_precision"]}
print("GOLD:", GOLD)

All gold IDs found in corpus.
GOLD: {0: {'astro-ph/9909356'}, 1: {'astro-ph/9905148'}, 2: {'1408.2479'}}


In [25]:
QUESTIONS = [e["question"] for e in EVAL_SET]
REFERENCE = ["20%", "-1 to 1.8", "from $10^{-4}$ to 0.1", None, None]

4. Query and evaluate retrieval: 

In [26]:
def norm_id(x):
    """Strip the version suffix: 'astro-ph/9909356v1' -> 'astro-ph/9909356'."""
    return re.sub(r"v\d+$", "", x)
def run_retrieval(k=3):
    rows = []
    for qi, q in enumerate(QUESTIONS):
        hits = retrieve(q, corpus, index, k=k)          # [(paper, score), ...]
        ids = [norm_id(p["id"]) for p, _ in hits]
        scores = [s for _, s in hits]
        p_at_3 = sum(i in GOLD[qi] for i in ids) / k if qi in GOLD else None
        rows.append({"q": qi + 1, "ids": ids, "scores": scores, "p@3": p_at_3})
        print(f"Q{qi+1}: {q}")
        for i, s in zip(ids, scores):
            print(f"   {i:<22} cos={s:.4f}")
        if p_at_3 is not None:
            print(f"   precision@3 = {p_at_3:.2f}")
    gold = [r["p@3"] for r in rows if r["p@3"] is not None]
    print(f"\nMean precision@3 over gold questions: {np.mean(gold):.3f}")
    return rows
retrieval_rows = run_retrieval(k=3)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Q1:  How much is the Bonnor-Ebert critical mass is reduced by when the helical fields are threaded throughout the core models?
   astro-ph/9909356       cos=0.7241
   0903.1810              cos=0.4428
   0912.3044              cos=0.4356
   precision@3 = 0.33


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Q2: How does the density distributions fall off as a function of radius for isothermal filament models with helical magnetic feilds?
   astro-ph/9905148       cos=0.5583
   astro-ph/9901096       cos=0.5452
   astro-ph/9909356       cos=0.5141
   precision@3 = 0.33


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Q3: When the magnetic field is initially uniform and aligned with the cloud rotation axes, what are the initial ratios of rotational energy to gravitational energy? 
   1501.07734             cos=0.5141
   1408.2479              cos=0.4953
   2405.19720             cos=0.4718
   precision@3 = 0.33


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Q4: How do Prolate cores with central densities 20 times higher than their boundary densities compare to prolate cores with central densities 100 times higher? 
   0903.1810              cos=0.4665
   0912.3044              cos=0.4393
   astro-ph/0206311       cos=0.4222


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Q5: What is the temperature range for the N49 bubble site?
   1711.06412             cos=0.4023
   1402.3033              cos=0.2736
   0912.3044              cos=0.2581

Mean precision@3 over gold questions: 0.333


Q1: Relevant. The top hit is the gold paper astro-ph/9909356 on prolate cores, and the other two are closely related core-collapse papers. The top cosine is 0.7241, well above the next two (0.4428 and 0.4356), so the gold paper stands out clearly.

Q2 Partially relevant. The gold paper astro-ph/9905148 is ranked first, with cosine 0.5583, but the margin is small. The ranking is fragile, so a small change in wording could reorder these. 

Q3: Partially relevant as the gold paper 1408.2479 is ranked second, with cosine 0.4953 a low value but still mentioned in the top 3 papers.

Q4: Partially Relevant as the top hit is 0903.1810, the gold paper listed in EVAL_SET for this question, with cosine 0.4665 and the other two (0.4393 and 0.4222) are on prolate cores. 

Q5: Partially Relevant but weak overall the the top hit, 1711.06412, is the gold N49 paper, but its cosine is only 0.4023.

In [27]:
results = []
for qi, q in enumerate(QUESTIONS):
    hits = retrieve(q, corpus, index, k=3)
    retrieved_ids = [norm_id(p["id"]) for p, _ in hits]
    context = "\n\n".join(f"[{norm_id(p['id'])}] {p['title']}\n{p['abstract']}"
                          for p, _ in hits)

    baseline = query_llm(q)
    rag = query_llm(q, system=f"Context:\n{context}\n\nAnswer using only the context above. Cite arXiv IDs in square brackets.")

    results.append({"q": qi + 1, "question": q, "retrieved": retrieved_ids,
                    "baseline": baseline, "rag": rag, "reference": REFERENCE[qi]})
    print(f"Q{qi+1}: {q}\nRetrieved: {retrieved_ids}\nBASELINE: {baseline}\nRAG: {rag}\n{'-'*60}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Q1:  How much is the Bonnor-Ebert critical mass is reduced by when the helical fields are threaded throughout the core models?
Retrieved: ['astro-ph/9909356', '0903.1810', '0912.3044']
BASELINE: # Bonnor-Ebert Mass with Helical Fields

I don't have specific quantitative data on the exact reduction factor for the Bonnor-Ebert critical mass when helical magnetic fields are threaded through isothermal core models.

However, I can provide context:

## General Effects of Magnetic Fields

- **Magnetic support** generally increases the critical mass needed for gravitational collapse, since magnetic pressure provides additional support against gravity
- This would **increase** the Bonnor-Ebert mass rather than reduce it

## Possible Interpretation

If you're referring to studies showing a *reduction* in critical mass, this might involve:

1. **Field geometry effects** - Certain field configurations might reduce support in specific directions
2. **Ambipolar diffusion** - Neutral-ion decoupling 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Q2: How does the density distributions fall off as a function of radius for isothermal filament models with helical magnetic feilds?
Retrieved: ['astro-ph/9905148', 'astro-ph/9901096', 'astro-ph/9909356']
BASELINE: # Density Profiles in Isothermal Filaments with Helical Magnetic Fields

## Standard Isothermal Filament (Non-magnetic)

For comparison, the classic **non-magnetic isothermal filament** follows:

$$\rho(r) \propto \frac{1}{r^2}$$

This comes from solving the hydrostatic equilibrium equation with constant temperature.

## With Helical Magnetic Fields

The addition of helical magnetic fields modifies this profile:

### Key Effects:

**1. Inner regions (r → 0):**
- The profile remains approximately **ρ ∝ r⁻²** 
- Magnetic support is relatively weak compared to pressure gradient

**2. Outer regions (r → R):**
- The density falls off **more steeply** than r⁻²
- The helical field provides **additional outward pressure** (magnetic pressure from field tension)
- Typically transition

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Q3: When the magnetic field is initially uniform and aligned with the cloud rotation axes, what are the initial ratios of rotational energy to gravitational energy? 
Retrieved: ['1501.07734', '1408.2479', '2405.19720']
BASELINE: # Rotational to Gravitational Energy Ratio in Magnetized Clouds

## The Classical Result

When a magnetic field is **initially uniform and aligned with the rotation axis** of a collapsing molecular cloud, the initial ratio of rotational energy to gravitational energy is:

$$\frac{E_{\text{rot}}}{|E_{\text{grav}}|} \approx \beta_{\text{rot}}$$

where **β_rot is the rotational parameter** (typically a small value).

## Typical Values

For realistic molecular clouds:

- **β_rot ~ 0.01 to 0.1** (or about 1-10%)

This means:
$$E_{\text{rot}} \ll |E_{\text{grav}}|$$

Rotational energy is typically **much smaller** than gravitational binding energy.

## Physical Significance

This configuration is important because:

1. **Magnetic field aligned with rotation**: The fi

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Q4: How do Prolate cores with central densities 20 times higher than their boundary densities compare to prolate cores with central densities 100 times higher? 
Retrieved: ['0903.1810', '0912.3044', 'astro-ph/0206311']
BASELINE: # Comparison of Prolate Cores at Different Density Contrasts

These two density profiles represent significantly different physical structures:

## **Central Density Contrast**

| Feature | 20× Contrast | 100× Contrast |
|---------|--------------|---------------|
| **Density ratio** | ρ_center/ρ_boundary | Much steeper gradient |
| **Profile shape** | Gentler, more gradual decline | Sharper, more pronounced peak |

## **Key Structural Differences**

### **Mass Distribution**
- **20× contrast**: Mass more evenly distributed throughout the core
- **100× contrast**: Mass heavily concentrated in the central region; outer regions contribute minimally

### **Physical Implications**

| Aspect | 20× | 100× |
|--------|-----|------|
| **Gravitational potential** | More 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Q5: What is the temperature range for the N49 bubble site?
Retrieved: ['1711.06412', '1402.3033', '0912.3044']
BASELINE: I don't have specific information about the temperature range for the "N49 bubble site" in my current knowledge base. 

To help you better, could you provide additional context? For example:
- Is this related to a specific research location or study?
- Is it associated with hydrothermal vents, geological research, or another field?
- Do you have a source or publication where you encountered this term?

With more details, I may be able to help you find the information you're looking for.
RAG: According to the context, the Herschel temperature map traces the structures in the N49 bubble site in a temperature range of approximately **16-24 K** [1711.06412].
------------------------------------------------------------


5. ANALYSIS COMPARISON

Q1: RAG better. RAG returned the reference value of about 20% and cited astro-ph/9909356, the top retrieved paper. The baseline gave no number and wrongly suggested magnetic support would increase the critical mass not considering the nuances of toroidal vs poloidal magnetic feilds.

Q2: RAG better, RAG’s “r^{-1.8} to r^{-2}” is close to that, but it cites astro-ph/9901096 the second paper rather than the gold paper 9905148, which was retrieved first.

Q3: RAG better. RAG gave the reference range of 10⁻⁴ to 0.1 from the gold paper 1408.2479. The baseline gave a generic 0.01–0.1 estimate with no source.

RAG improved all gold questions with claude-haiku-4-5-20251001 LLM. For "How much is the Bonnor-Ebert critical mass is reduced by when the helical fields are threaded throughout the core models?", the gold paper (astro-ph/9909356) came up first in retrieval, and RAG gave the reference answer of about 20%. The baseline didn't give a number at all. For "When the magnetic field is initially uniform and aligned with the cloud rotation axes, what are the initial ratios of rotational energy to gravitational energy?", the gold paper 1408.2479 was second, and RAG returned the reference range of 10⁻⁴ to 0.1, which the baseline couldn't match. For the filament density question, the gold paper astro-ph/9905148 beat the next paper by only 0.013 cosine, and while it returned the correct value: "r^{-1.8} to r^{-2},  the wording has a massive effect on cosine-based ranking variability. 